In [ ]:
import pandas as pd
import numpy as np

def satisfies(example, hypothesis):
    return all(h == '?' or h == val for h, val in zip(hypothesis, example))

def is_more_general(g, s):
    return all(g_i == '?' or g_i == s_i for g_i, s_i in zip(g, s))

def candidate_elimination(csv_file_path):
    df = pd.read_csv(csv_file_path)
    concepts = np.array(df.iloc[:, :-1])
    target = np.array(df.iloc[:, -1])

    num_attributes = concepts.shape[1]

    S = ['0'] * num_attributes
    G = [['?'] * num_attributes]

    for i, t in enumerate(target):
        if t == 'Yes':
            S = list(concepts[i])
            break

    print("Initial S_0:", S)
    print("Initial G_0:", G)
    print("-" * 50)

    for i, example in enumerate(concepts):
        if target[i] == 'Yes':

            G = [g for g in G if satisfies(example, g)]

            for j in range(num_attributes):
                if S[j] != example[j]:
                    S[j] = '?'

        elif target[i] == 'No':

            G_new = []
            for g in G:
                if satisfies(example, g):
                    for j in range(num_attributes):
                        if g[j] == '?':
                            if S[j] != example[j] and S[j] != '?':
                                g_spec = g.copy()
                                g_spec[j] = S[j]
                                G_new.append(g_spec)
                else:
                    G_new.append(g)

            G = [g for g in G_new if is_more_general(g, S)]

        print(f"After Sample {i+1} ({target[i]}):")
        print("S =", S)
        print("G =", G)
        print("-" * 50)

    return S, G

csv_content = """Temp,Milk,Flavor,Enjoyed
Hot,Oat,Vanilla,Yes
Hot,Whole,Hazelnut,No
Iced,Oat,Vanilla,Yes"""

with open("coffee_dataset.csv", "w") as f:
    f.write(csv_content)

print("=== Candidate-Elimination Algorithm Trace ===")
final_S, final_G = candidate_elimination("coffee_dataset.csv")

print("=" * 50)
print("Final Specific Boundary (S):", final_S)
print("Final General Boundary (G) :", final_G)

=== Candidate-Elimination Algorithm Trace ===
Initial S_0: ['Hot', 'Oat', 'Vanilla']
Initial G_0: [['?', '?', '?']]
--------------------------------------------------
After Sample 1 (Yes):
S = ['Hot', 'Oat', 'Vanilla']
G = [['?', '?', '?']]
--------------------------------------------------
After Sample 2 (No):
S = ['Hot', 'Oat', 'Vanilla']
G = [['?', 'Oat', '?'], ['?', '?', 'Vanilla']]
--------------------------------------------------
After Sample 3 (Yes):
S = ['?', 'Oat', 'Vanilla']
G = [['?', 'Oat', '?'], ['?', '?', 'Vanilla']]
--------------------------------------------------
Final Specific Boundary (S): ['?', 'Oat', 'Vanilla']
Final General Boundary (G) : [['?', 'Oat', '?'], ['?', '?', 'Vanilla']]
